# SWAN Nested Grid: Model Setup & Execution

---

## Overview

This notebook configures and runs a **two-stage nested SWAN simulation** for the California Bay case study. The nested approach provides high-resolution wave fields in the WEC deployment zone while keeping computational cost manageable.

### Workflow

```
Stage 1: Coarse (Parent) Run
  └─ Full California Bay domain (120×216, ~460 m)
  └─ Forced by ERA5 wind + TPAR wave boundary
  └─ NESTOUT: saves spectral boundary along fine grid edges
  └─ Output: nest_boundary.bnd + coarse_block.mat

Stage 2: Fine (Nested) Run
  └─ High-energy deployment zone (300×300, ~100 m)
  └─ NESTINP: reads spectral boundary from Stage 1
  └─ Same wind forcing and physics as coarse grid
  └─ Output: fine_block.mat (hourly, high-resolution)
```

### Key SWAN Commands for Nesting

| Command | Stage | Purpose |
|---------|-------|--------|
| `NGRID` | Both | Define the nested grid boundary geometry |
| `NESTOUT` | Coarse | Write full spectral data along the nested grid boundary |
| `NESTINP` | Fine | Read spectral boundary conditions from the coarse run |

### Important Notes

- **One-way nesting**: Information flows from coarse → fine only. The fine grid does not feed back.
- **`NGRID` must match**: The `NGRID` definition must be identical in both input files (same origin, extent, and resolution).
- **Physics consistency**: Both grids should use the same physics settings to avoid discontinuities at the nesting boundary.
- **Temporal resolution**: The `NESTOUT` interval should match or exceed the fine grid's needs (we use 1 HR).

### Prerequisites

- SWAN must be installed and available on PATH
- `swan_runner.py` module (in `code/`)
- California Bay data files (GEBCO, wind, boundary conditions)

### Dependencies

```
numpy, scipy, matplotlib
```

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Locate repository root robustly (works regardless of kernel cwd)
# --- bootstrap: locate the repository and make code/ importable ---
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO_ROOT, 'code')):
    if os.path.dirname(REPO_ROOT) == REPO_ROOT:
        raise FileNotFoundError('Repository root not found (no code/ directory). '
                                'Start Jupyter from inside the repository.')
    REPO_ROOT = os.path.dirname(REPO_ROOT)
CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# --- end bootstrap ---
import swan_runner
from regions import get_region, summarize  # noqa: E402

# Region pick — only regions that define a nested grid will run the full
# workflow; others get a friendly skip.
REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
HAS_NESTED = cfg.nested_subdir is not None
print(summarize(REGION))

MODEL_DIR = cfg.nested_dir if HAS_NESTED else cfg.model_dir

SWAN_CMD = os.environ.get('SWAN_CMD', 'swanrun')
SWAN_NPROC = int(os.environ.get('SWAN_NPROC', '6'))

print(f"Model directory : {MODEL_DIR}")
print(f"SWAN command    : {SWAN_CMD}  (available: {swan_runner.check_swan_available(SWAN_CMD)})")
print(f"Has nested conf : {HAS_NESTED}")
if not HAS_NESTED:
    print()
    print(f"NOTE: region {cfg.display_name!r} does not have a nested SWAN setup in "
          "this repository. The notebook will only run the single-grid stage below.")

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('02_model_setup')
print(f'Figures will be saved to: {FIGS_DIR}')


## 1. Review SWAN Input Files

### 1.1 Coarse Grid Input (`INPUT_coarse`)

The coarse input is identical to the original California Bay model, with two additions:
1. **`NGRID` + `NESTOUT`**: Defines the nested grid boundary and writes spectral data along it
2. **`BLOCK` output**: Saves the 2-D wave field for comparison with the fine grid

The `NGRID` command specifies the geometry of the nested boundary:
```
NGRID 'nestzn' xpc ypc alpc xlenc ylenc mxn myn
```
where `mxn` and `myn` define how many points SWAN uses to discretize the boundary (not the fine grid's computational cells).

In [ ]:
if HAS_NESTED:
    coarse_input = os.path.join(MODEL_DIR, cfg.nested_coarse_input)
    label = 'INPUT_coarse'
else:
    coarse_input = cfg.swan_input_path
    label = f'{cfg.swan_input_name}  (single-grid fallback)'

print("=" * 65)
print(label)
print("=" * 65)
if os.path.exists(coarse_input):
    with open(coarse_input) as f:
        print(f.read())
else:
    print(f"ERROR: file not found: {coarse_input}")


### 1.2 Fine Grid Input (`INPUT_fine`)

The fine grid input has a different `CGRID` (higher resolution, smaller domain) and replaces the western boundary condition (`BOUndspec SIDE W`) with `NESTINP`, which reads the spectral boundary from the coarse run.

Key differences from the coarse input:
- `CGRID`: 300×300 cells at ~100 m resolution, no rotation
- `NESTINP` instead of `BOUndspec`: Boundary comes from the coarse run, not ERA5
- `BLOCK` output at **hourly** intervals (vs 6-hourly for coarse) to capture detailed temporal variability

In [ ]:
if HAS_NESTED:
    fine_input = os.path.join(MODEL_DIR, cfg.nested_fine_input)
    print("=" * 65)
    print("INPUT_fine")
    print("=" * 65)
    if os.path.exists(fine_input):
        with open(fine_input) as f:
            print(f.read())
    else:
        print(f"ERROR: file not found: {fine_input}")
else:
    print("No nested (fine) grid configured for this region.")


## 2. Stage 1: Run the Coarse (Parent) Grid

The coarse run solves the spectral action balance equation over the full California Bay domain. At each output timestep, SWAN writes the 2-D directional wave spectrum along the fine grid boundary to `nest_boundary.bnd`.

This file contains the full spectral information (energy density as a function of frequency and direction) — not just bulk parameters like $H_s$ and $T_p$. This is critical because the fine grid needs the complete spectral shape to properly account for swell and wind-sea components arriving from different directions.

**Expected runtime**: Several minutes depending on your system (the grid has 25,920 cells × 1440 timesteps).

In [ ]:
if not swan_runner.check_swan_available(SWAN_CMD):
    print(f"WARNING: '{SWAN_CMD}' not found on PATH.")
    print("Install SWAN and re-run. See the repository README.")
else:
    print("=" * 65)
    if HAS_NESTED:
        print("Stage 1: Running COARSE (parent) grid...")
        input_name = cfg.nested_coarse_input
    else:
        print(f"Single-grid stage: running {cfg.swan_input_name} "
              f"for {cfg.display_name}")
        input_name = cfg.swan_input_name
    print("=" * 65)

    result = swan_runner.run_swan(
        MODEL_DIR, input_name=input_name,
        swan_cmd=SWAN_CMD, nproc=SWAN_NPROC,
    )

    if HAS_NESTED:
        nest_file = os.path.join(MODEL_DIR, 'nest_boundary.bnd')
        if os.path.exists(nest_file):
            size_mb = os.path.getsize(nest_file) / 1e6
            print(f"\n  nest_boundary.bnd created: {size_mb:.1f} MB")
        else:
            print("\n  WARNING: nest_boundary.bnd not found!")
        coarse_block = os.path.join(MODEL_DIR, 'coarse_block.mat')
        if os.path.exists(coarse_block):
            print(f"  coarse_block.mat created: "
                  f"{os.path.getsize(coarse_block)/1e6:.1f} MB")
    print("\n  Stage complete.")


## 3. Stage 2: Run the Fine (Nested) Grid

The fine grid reads the spectral boundary from `nest_boundary.bnd` and runs at ~100 m resolution over the deployment zone. The boundary conditions are applied along all four edges of the fine grid, providing spatially and temporally varying wave input from the coarse solution.

Because the fine grid has many more cells per unit area ($300 \times 300 = 90{,}000$ cells), it may take longer than the coarse run despite covering a smaller domain. The hourly output interval also increases the output file size.

**Expected runtime**: Several minutes (90,000 cells × 1440 timesteps).

In [ ]:
if not HAS_NESTED:
    print(f"Region {cfg.display_name!r} has no nested configuration — "
          "Stage 2 is skipped.")
else:
    nest_file = os.path.join(MODEL_DIR, 'nest_boundary.bnd')
    if not swan_runner.check_swan_available(SWAN_CMD):
        print(f"WARNING: '{SWAN_CMD}' not found on PATH.")
    elif not os.path.exists(nest_file):
        print("WARNING: nest_boundary.bnd not found. Run Stage 1 first.")
    else:
        print("=" * 65)
        print("Stage 2: Running FINE (nested) grid...")
        print("=" * 65)
        result = swan_runner.run_swan(
            MODEL_DIR, input_name=cfg.nested_fine_input,
            swan_cmd=SWAN_CMD, nproc=SWAN_NPROC,
        )
        fine_block = os.path.join(MODEL_DIR, 'fine_block.mat')
        if os.path.exists(fine_block):
            print(f"\n  fine_block.mat created: "
                  f"{os.path.getsize(fine_block)/1e6:.1f} MB")
        print("\n  Stage 2 complete.")


## 4. Compare Results: Coarse vs Fine Grid

After both runs complete, we compare the wave fields from the coarse and fine grids over the nested domain. This reveals the additional detail that the fine grid provides:
- Sharper refraction patterns
- Better-resolved wave shadow zones
- More precise energy hotspot locations

We read both `.MAT` block outputs and overlay them.

In [ ]:
import warnings
if not HAS_NESTED:
    print(f"No nested grid for {cfg.display_name!r} — "
          "skipping coarse/fine comparison.")
else:
    coarse_block_file = os.path.join(MODEL_DIR, 'coarse_block.mat')
    fine_block_file = os.path.join(MODEL_DIR, 'fine_block.mat')
    has_coarse = os.path.exists(coarse_block_file)
    has_fine = os.path.exists(fine_block_file)
    print(f"Coarse block output: {'OK' if has_coarse else 'MISSING (run Stage 1)'}")
    print(f"Fine block output:   {'OK' if has_fine else 'MISSING (run Stage 2)'}")

    if has_coarse and has_fine:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mat_c = swan_runner.read_mat_block(coarse_block_file)
        Xp_c, Yp_c = mat_c['Xp'], mat_c['Yp']
        Hs_c = swan_runner.extract_block_variable(mat_c, 'Hsig')
        Hs_c = np.where(Hs_c < 0, np.nan, Hs_c)
        Hs_mean_c = np.nanmean(Hs_c, axis=0)
        print(f"\nCoarse grid: {Xp_c.shape[1]}x{Xp_c.shape[0]}, {Hs_c.shape[0]} timesteps")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mat_f = swan_runner.read_mat_block(fine_block_file)
        Xp_f, Yp_f = mat_f['Xp'], mat_f['Yp']
        Hs_f = swan_runner.extract_block_variable(mat_f, 'Hsig')
        Hs_f = np.where(Hs_f < 0, np.nan, Hs_f)
        Hs_mean_f = np.nanmean(Hs_f, axis=0)
        print(f"Fine grid:   {Xp_f.shape[1]}x{Xp_f.shape[0]}, {Hs_f.shape[0]} timesteps")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        vmin = 0
        vmax = max(np.nanmax(Hs_mean_f), np.nanpercentile(Hs_mean_c, 99))
        levels = np.linspace(vmin, vmax, 25)
        fine_lon = (Xp_f.min(), Xp_f.max())
        fine_lat = (Yp_f.min(), Yp_f.max())

        land_c = np.isnan(Hs_mean_c)
        cf1 = ax1.contourf(Xp_c, Yp_c, np.ma.masked_where(land_c, Hs_mean_c),
                           levels=levels, cmap='YlOrRd', extend='both')
        ax1.set_xlim(fine_lon); ax1.set_ylim(fine_lat)
        ax1.set_title('Coarse Grid', fontsize=12, fontweight='bold')
        ax1.set_aspect('equal'); ax1.grid(True, alpha=0.15, ls='--')
        plt.colorbar(cf1, ax=ax1, shrink=0.8, label='Mean $H_s$ (m)')

        land_f = np.isnan(Hs_mean_f)
        cf2 = ax2.contourf(Xp_f, Yp_f, np.ma.masked_where(land_f, Hs_mean_f),
                           levels=levels, cmap='YlOrRd', extend='both')
        ax2.set_title('Fine Nested Grid', fontsize=12, fontweight='bold')
        ax2.set_aspect('equal'); ax2.grid(True, alpha=0.15, ls='--')
        plt.colorbar(cf2, ax=ax2, shrink=0.8, label='Mean $H_s$ (m)')
        fig.suptitle(f"Nested Grid Comparison — {cfg.display_name}",
                     fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        _fname = f"nested_setup_{cfg.key}_cell11.png"
        plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
        plt.show()

        print(f"\nStatistics over fine grid domain:")
        print(f"  Coarse: mean Hs = {np.nanmean(Hs_mean_c):.3f} m, max = {np.nanmax(Hs_mean_c):.3f} m")
        print(f"  Fine:   mean Hs = {np.nanmean(Hs_mean_f):.3f} m, max = {np.nanmax(Hs_mean_f):.3f} m")
    else:
        print("\nRun both stages to compare results.")


## 5. Wave Power Map on Fine Grid

The fine grid provides the high-resolution wave power density field that is most useful for WEC array optimization. This is the field that the GA optimizer (notebook `04_optimization/`) would use for the coupled optimization.

In [ ]:
if not HAS_NESTED:
    print(f"No nested grid for {cfg.display_name!r} — skipping fine-grid power map.")
else:
    fine_block_file = os.path.join(MODEL_DIR, 'fine_block.mat')
    if not os.path.exists(fine_block_file):
        print("Fine grid output not available. Run Stage 2 first.")
    else:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mat_f = swan_runner.read_mat_block(fine_block_file)
        Xp_f, Yp_f = mat_f['Xp'], mat_f['Yp']
        Hs_f = swan_runner.extract_block_variable(mat_f, 'Hsig')
        Hs_f = np.where(Hs_f < 0, np.nan, Hs_f)
        Hs_mean_f = np.nanmean(Hs_f, axis=0)
        Tp_f = swan_runner.extract_block_variable(mat_f, 'RTpeak')
        Tp_f = np.where(Tp_f < 0, np.nan, Tp_f)
        Tp_mean_f = np.nanmean(Tp_f, axis=0)

        rho_g2 = 1025 * 9.81**2 / (64 * np.pi)
        P_fine = rho_g2 * Hs_mean_f**2 * Tp_mean_f / 1000
        land_f = np.isnan(Hs_mean_f)

        fig, ax = plt.subplots(figsize=(12, 10))
        vmax_p = np.nanpercentile(P_fine[~land_f], 99)
        levels_p = np.linspace(0, vmax_p, 25)
        cf = ax.contourf(Xp_f, Yp_f, np.ma.masked_where(land_f, P_fine),
                         levels=levels_p, cmap='inferno', extend='max')
        plt.colorbar(cf, ax=ax, shrink=0.8, label='Mean Wave Power Density (kW/m)')
        ax.set_title(f'Wave Power Density — Fine Grid — {cfg.display_name}',
                     fontsize=13, fontweight='bold')
        ax.set_aspect('equal'); ax.grid(True, alpha=0.15, ls='--')
        plt.tight_layout(); _fname = f"nested_setup_{cfg.key}_cell13.png"
        plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
        plt.show()

        print(f"Max:  {np.nanmax(P_fine):.1f} kW/m")
        print(f"Mean: {np.nanmean(P_fine[~land_f]):.1f} kW/m")


## Summary

This notebook demonstrated the complete SWAN nested grid workflow:

| Stage | Grid | Resolution | Key Output |
|-------|------|-----------|------------|
| 1 (Coarse) | 120×216 | ~460 m | `nest_boundary.bnd` (spectral BC) + `coarse_block.mat` |
| 2 (Fine) | 300×300 | ~100 m | `fine_block.mat` (high-res wave field) |

### SWAN Nesting Commands Reference

```
! In the COARSE input file:
NGRID 'name' xpc ypc alpc xlenc ylenc mxn myn
NESTOUT 'name' 'filename.bnd' OUTPUT tbegin dt_unit

! In the FINE input file:
BOUNdnest1 NESTing 'filename.bnd' CLOSed
```

### When to Use Nesting

- **WEC site assessment**: High resolution around deployment sites for accurate power estimates
- **Coastal engineering**: Detailed wave conditions at harbor entrances, breakwaters
- **Sediment transport**: Fine-scale wave-driven currents in nearshore zones
- **Storm impact studies**: Detailed overtopping and inundation at specific locations

### Limitations

- **One-way only**: Processes in the fine grid (e.g., WEC wave absorption) don't affect the coarse grid
- **Bathymetry resolution**: GEBCO (~460 m) limits the fine grid's effective resolution; for true 100 m physics, higher-resolution bathymetry is needed
- **Computational cost**: The fine grid has ~3.5× more cells than the coarse grid, plus hourly output
- **Spectral boundary file size**: `nest_boundary.bnd` can be large (hundreds of MB to GB) for long simulations with many spectral bins